# RoBERTa Hate Speech Inference

1. **Zero-shot** — `roberta-large-mnli` (no hate speech training, uses NLI to classify)
2. **Fine-tuned** — `shareeep/roberta-base-fine-tuned-hatexplain`

In [11]:
from transformers import pipeline

LABELS = ["hate speech", "normal", "offensive"]

# roberta-large-mnli: RoBERTa trained on Multi-NLI (natural language inference)
# No hate-speech-specific training — uses entailment to estimate label fit
zero_shot = pipeline("zero-shot-classification", model="roberta-large-mnli")
print("✅ Zero-shot pipeline ready (roberta-large-mnli)")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1746.10it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Zero-shot pipeline ready (roberta-large-mnli)


In [16]:
# === Enter your text here ===
TEXT = "These malay fellas damn CB kia. Only know how to make noise and cause trouble."

result = zero_shot(TEXT, candidate_labels=LABELS)
print(f"Text: {TEXT}\n")
for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 40)
    print(f"  {label:12s} {score:.4f} {bar}")
print(f"\nPrediction: {result['labels'][0]}")

Text: These malay fellas damn CB kia. Only know how to make noise and cause trouble.

  offensive    0.5774 ███████████████████████
  hate speech  0.3596 ██████████████
  normal       0.0631 ██

Prediction: offensive


In [ ]:
## Fine-tuned Model: shareeep/roberta-base-fine-tuned-hatexplain

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

FT_MODEL = "shareeep/roberta-base-fine-tuned-hatexplain"

ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL)
ft_model = AutoModelForSequenceClassification.from_pretrained(FT_MODEL)
ft_model.eval()

# Show the label mapping from the model config
id2label = ft_model.config.id2label
print("✅ Fine-tuned model loaded:", FT_MODEL)
print("Labels:", id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1764.67it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              

✅ Fine-tuned model loaded: shareeep/roberta-base-fine-tuned-hatexplain
Labels: {0: 'hatespeech', 1: 'normal', 2: 'offensive'}


In [17]:
# === Enter your text here ===
TEXT_FT = "These malay fellas damn CB kia. Only know how to make noise and cause trouble."

inputs = ft_tokenizer(TEXT_FT, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    logits = ft_model(**inputs).logits

probs = torch.softmax(logits, dim=-1).squeeze()
ranked = sorted(enumerate(probs.tolist()), key=lambda x: x[1], reverse=True)

print(f"Text: {TEXT_FT}\n")
for idx, prob in ranked:
    label = id2label.get(idx, f"label_{idx}")
    bar = "█" * int(prob * 40)
    print(f"  {label:12s} {prob:.4f} {bar}")

print(f"\nPrediction: {id2label.get(ranked[0][0], ranked[0][0])}")

Text: These malay fellas damn CB kia. Only know how to make noise and cause trouble.

  normal       0.6916 ███████████████████████████
  offensive    0.2699 ██████████
  hatespeech   0.0386 █

Prediction: normal
